# Import libraries

In [ ]:
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col

# Read Customer profiles from silver layer

In [ ]:
df = spark.read.table("silver.crm.customers")
display(df.select("customerid","gender","tenure","monthlycharges","totalcharges","review", "sentimentScore"))

In [ ]:
df = df.drop("review")
df.printSchema()

# Load saved churn prediction model

In [ ]:
loaded_model = PipelineModel.load("/Workspace/model/customer_churn/ml_model/")

# Predict churn

In [ ]:
predictions = loaded_model.transform(df)

# View results

In [ ]:
predictions.select("customerid", "prediction").show()

# Column subsetting - get relevant columns

In [ ]:
selected_columns = [
    "customerid", 
    "gender", 
    "seniorcitizen", 
    "partner", 
    "dependents", 
    "tenure", 
    "phoneservice", 
    "multiplelines", 
    "internetservice", 
    "onlinesecurity", 
    "onlinebackup", 
    "deviceprotection", 
    "techsupport", 
    "streamingtv", 
    "streamingmovies", 
    "contract", 
    "paperlessbilling", 
    "paymentmethod", 
    "monthlycharges", 
    "totalcharges", 
    "sentimentScore",
    "prediction"
]
customers_selected = predictions.select(*[col(c) for c in selected_columns])
customers_selected.select("customerid","gender","tenure","monthlycharges","totalcharges", "sentimentScore", "prediction").show()

# Save results to ADW

In [ ]:
customers_selected.write.insertInto("gold.admin.customer_churn")